In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/climate-impact-assessment-policy-analysis/climate_change_impact_on_agriculture_2024.csv


In [2]:
import pandas as pd

# Chemin du fichier
file_path = "/kaggle/input/climate-impact-assessment-policy-analysis/climate_change_impact_on_agriculture_2024.csv"

# Chargement du fichier
df = pd.read_csv(file_path)

# Affichage des premières lignes pour inspection
df.head()

,Year,Country,Region,Crop_Type,Average_Temperature_C,Total_Precipitation_mm,CO2_Emissions_MT,Crop_Yield_MT_per_HA,Extreme_Weather_Events,Irrigation_Access_%,Pesticide_Use_KG_per_HA,Fertilizer_Use_KG_per_HA,Soil_Health_Index,Adaptation_Strategies,Economic_Impact_Million_USD
0,2001,India,West Bengal,Corn,1.55,447.06,15.22,1.737,8,14.54,10.08,14.78,83.25,Water Management,808.13
1,2024,China,North,Corn,3.23,2913.57,29.82,1.737,8,11.05,33.06,23.25,54.02,Crop Rotation,616.22
2,2001,France,Ile-de-France,Wheat,21.11,1301.74,25.75,1.719,5,84.42,27.41,65.53,67.78,Water Management,796.96
3,2001,Canada,Prairies,Coffee,27.85,1154.36,13.91,3.890,5,94.06,14.38,87.58,91.39,No Adaptation,790.32
4,1998,India,Tamil Nadu,Sugarcane,2.19,1627.48,11.81,1.080,9,95.75,44.35,88.08,49.61,Crop Rotation,401.72


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Encodage des stratégies d'adaptation en valeurs numériques
le = LabelEncoder()
df["Adaptation_Strategies_Encoded"] = le.fit_transform(df["Adaptation_Strategies"])

# Définition de la variable cible (rendement élevé ou faible)
median_yield = df["Crop_Yield_MT_per_HA"].median()
df["High_Yield"] = (df["Crop_Yield_MT_per_HA"] >= median_yield).astype(int)

# Sélection des caractéristiques et de la cible
X = df[["Adaptation_Strategies_Encoded", "Irrigation_Access_%", "Total_Precipitation_mm", "Average_Temperature_C"]]
y = df["High_Yield"]

# Division en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraînement du modèle de régression logistique
model = LogisticRegression()
model.fit(X_train, y_train)

# Prédictions et évaluation
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.582

The accuracy of the logistic regression model in predicting whether an adaptation strategy improves crop yield is approximately 58.2%. This suggests a relationship, but it could be refined with additional variables or methods.

In [4]:
import statsmodels.api as sm

# Sélection des variables indépendantes et dépendantes
X = df[["Average_Temperature_C", "Total_Precipitation_mm", "CO2_Emissions_MT", "Extreme_Weather_Events"]]
y = df["Economic_Impact_Million_USD"]

# Ajout de la constante pour le modèle
X = sm.add_constant(X)

# Régression linéaire
model = sm.OLS(y, X).fit()

# Résumé du modèle
print(model.summary())


                                 OLS Regression Results                                
Dep. Variable:     Economic_Impact_Million_USD   R-squared:                       0.042
Model:                                     OLS   Adj. R-squared:                  0.042
Method:                          Least Squares   F-statistic:                     109.3
Date:                         Thu, 20 Mar 2025   Prob (F-statistic):           2.67e-91
Time:                                 19:15:16   Log-Likelihood:                -74248.
No. Observations:                        10000   AIC:                         1.485e+05
Df Residuals:                             9995   BIC:                         1.485e+05
Df Model:                                    4                                         
Covariance Type:                     nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------

The temperature change is the most significant factor in agricultural economics. However, precipitation and extreme events show no clear effect. Maybe if I test nonlinear regression, Random Forest, and causal models, I could better understand the link. LET'S SEE !

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Charger le fichier CSV
df = pd.read_csv("/kaggle/input/climate-impact-assessment-policy-analysis/climate_change_impact_on_agriculture_2024.csv")

# Sélection des variables
X = df[["Average_Temperature_C", "Total_Precipitation_mm", "CO2_Emissions_MT", "Extreme_Weather_Events"]]
y = df["Economic_Impact_Million_USD"]

# Normalisation des variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Séparation des données en ensemble d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Modèle Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Prédictions
y_pred = rf_model.predict(X_test)

# Évaluation du modèle
r2_rf = r2_score(y_test, y_pred)
print(f"R² du modèle Random Forest : {r2_rf:.4f}")

# Importance des variables
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\nImportance des variables :")
print(importances.sort_values(ascending=False))


R² du modèle Random Forest : 0.1216

Importance des variables :
Average_Temperature_C     0.392926
Total_Precipitation_mm    0.254381
CO2_Emissions_MT          0.244586
Extreme_Weather_Events    0.108108
dtype: float64


The Random Forest model's R² is 0.1216, indicating a weak fit. The most important variable is Average Temperature (0.3929), followed by Precipitation (0.2544) and CO2 Emissions (0.2446), while Extreme Weather Events (0.1081) has the least importance. The next step is to Improve the model by adding other variables, using more suitable modeling techniques (e.g., nonlinear regression or causal models), or adjusting the model's parameters.

The Random Forest model's R² is 0.1216, indicating a weak fit. The most important variable is Average Temperature (0.3929), followed by Precipitation (0.2544) and CO2 Emissions (0.2446), while Extreme Weather Events (0.1081) has the least importance. The next step is to Improve the model by adding other variables, using more suitable modeling techniques (e.g., nonlinear regression or causal models), or adjusting the model's parameters. Let's do it tommorow.